# Monte Carlo Stock Simulation
## Every formula explained: purpose + every symbol

This notebook is an **explanatory companion** to a Monte Carlo stock simulation.

Goal:
- explain **each formula**, 
- explain the **purpose of the formula**, and
- explain **every symbol** inside it.

All math is written in **LaTeX**.


---
## 1) Log return

### Formula
$$
\ell_t = \ln\left(\frac{S_t}{S_{t-1}}\right)
$$

### Purpose (why we use it)
Stock prices evolve **multiplicatively**. Log returns convert this to an **additive** quantity. Additive quantities are easier to simulate and to sum across time.

### Symbols
- $\ell_t$ : log return at time (day) $t$.
- $\ln(\cdot)$ : natural logarithm (base $e$).
- $S_t$ : price at day $t$.
- $S_{t-1}$ : price at previous day.
- $\frac{S_t}{S_{t-1}}$ : multiplicative price change factor.

### Where it goes in the code
We compute historical $\ell_t$ from price history and then:
- **GBM** uses its estimated mean/volatility ($\mu,\sigma$)
- **Bootstrap** samples directly from historical $\ell_t$


---
## 2) Additivity across many days

### Formula
$$
\ln\left(\frac{S_T}{S_0}\right) = \sum_{t=1}^{T} \ell_t
$$

### Purpose
This is the **key identity** behind Monte Carlo with log returns. It says: the log of the total price change from day 0 to day $T$ equals the **sum** of daily log returns.

### Symbols
- $S_0$ : initial price (today / start of simulation).
- $S_T$ : price after $T$ days.
- $T$ : horizon (number of simulated days).
- $\sum_{t=1}^{T}$ : sum over days 1..T.
- $\ell_t$ : daily log return.

### Where it goes in the code
We implement the sum with a cumulative sum:
`log_paths = shocks.cumsum(axis=0)`


---
## 3) Converting log returns back to prices

### Formula
$$
S_T = S_0\,\exp\left(\sum_{t=1}^{T} \ell_t\right)
$$

### Purpose
After we simulate (or resample) daily log returns, we need actual prices. This formula converts the summed log returns back into a price.

### Symbols
- $\exp(\cdot)$ : exponential function ($e^{x}$), the inverse of $\ln$.
- $\sum_{t=1}^{T} \ell_t$ : accumulated log return.
- $S_0$ : starting price.
- $S_T$ : price at horizon.

### Where it goes in the code
`prices = S0 * np.exp(log_paths)`


---
## 4) GBM-style assumption: normal log returns

### Formula
$$
\ell_t \sim \mathcal{N}(\mu, \sigma)
$$

### Purpose
GBM assumes daily log returns behave like draws from a normal distribution. This lets us generate new random returns using only two parameters.

### Symbols
- $\sim$ : "is distributed as".
- $\mathcal{N}(\mu,\sigma)$ : normal distribution.
- $\mu$ : mean daily log return (drift).
- $\sigma$ : standard deviation of daily log returns (volatility).

### Where it goes in the code
`shocks = rng.normal(loc=mu, scale=sigma, size=(horizon, sims))`


---
## 5) Estimating $\mu$ and $\sigma$ from history

### Formulas
$$
\mu = \mathbb{E}[\ell_t]
$$
$$
\sigma = \sqrt{\mathbb{V}[\ell_t]}
$$

### Purpose
To run GBM, we need parameters of the return distribution. We estimate them from historical log returns.

### Symbols
- $\mathbb{E}[\cdot]$ : expectation (mean).
- $\mathbb{V}[\cdot]$ : variance.
- $\sqrt{\cdot}$ : square root.
- $\mu$ : estimated average daily log return.
- $\sigma$ : estimated volatility.

### Where it goes in the code
In Python this is:
- `mu = log_returns.mean()`
- `sigma = log_returns.std(ddof=1)`


---
## 6) Bootstrap model: resampling historical returns

### Conceptual formula
$$
\ell_t \sim \{\ell_1^{hist}, \ell_2^{hist}, \dots, \ell_N^{hist}\}
$$

### Purpose
Bootstrap avoids the normality assumption. Instead, it samples returns from the **empirical distribution** observed in the past. This preserves skewness and fat tails (if present historically).

### Symbols
- $\ell_t$ : simulated log return.
- $\ell_i^{hist}$ : historical log return at index $i$.
- $N$ : number of historical return observations.
- $\{\cdot\}$ : the set (or collection) of historical returns.

### Where it goes in the code
Simple bootstrap is:
`shocks = rng.choice(log_returns, size=(horizon, sims), replace=True)`


---
## 7) Block bootstrap: preserving short-term dependence

### Idea
Instead of sampling single days independently, we sample blocks of consecutive returns.

### Purpose
Markets often show short-term patterns like volatility clustering. Blocks can preserve some local structure better than i.i.d. sampling.

### Symbols
- $b$ : block size (number of consecutive days).
- $\ell_{k:k+b-1}^{hist}$ : a slice (block) of historical returns.

### Where it goes in the code
We repeatedly pick a random `start_idx` and copy `block` returns into the scenario until the horizon is filled.


---
## 8) The paths matrix (main output)

### Formula
$$
\text{paths} \in \mathbb{R}^{(T+1) \times N}
$$
$$
\text{paths}[t,s] = S_t^{(s)}
$$

### Purpose
This matrix stores **all simulated prices**.

### Symbols
- $T$ : horizon (days).
- $N$ : number of scenarios (simulations).
- $t$ : time index (0..T).
- $s$ : scenario index (1..N).
- $S_t^{(s)}$ : price at time $t$ in scenario $s$.

### Where it goes in the code
- `paths[0, :]` is set to $S_0$ (the first row).
- `paths[-1, :]` is the final distribution.
- quantiles are computed across scenarios for each day.


---
## 9) Quantiles (fan chart)

### Definition
$$
Q(q) = \inf\{x : F(x) \ge q\}
$$

### Purpose
Quantiles summarize uncertainty. For example:
- $q=0.05$ means 5% of scenarios are below that value.
- $q=0.50$ is the median.

### Symbols
- $Q(q)$ : q-quantile.
- $q$ : probability level (e.g. 0.05, 0.50, 0.95).
- $\inf$ : infimum (smallest x that satisfies the condition).
- $F(x)$ : cumulative distribution function (CDF).

### Where it goes in the code
`np.quantile(paths, q=qs, axis=1)` computes quantiles **across scenarios** for each time step.


---
## 10) Final return (percentage change)

### Formula
$$
\Delta = \frac{S_T - S_0}{S_0}
$$

### Purpose
This converts the final price into a relative return (percentage change). It is used for scenario classification and histogram plots.

### Symbols
- $\Delta$ : relative return over the horizon.
- $S_T$ : final price.
- $S_0$ : initial price.

### Where it goes in the code
`pct_changes = (final_prices - S0) / S0`


---
## 11) Growth / Flat / Decline buckets

Let $\varepsilon$ be the flat threshold (in code: `FLAT_PCT`).

### Formulas
$$
\text{Growth if } \Delta > \varepsilon
$$
$$
\text{Flat if } |\Delta| \le \varepsilon
$$
$$
\text{Decline if } \Delta < -\varepsilon
$$

### Purpose
These buckets convert the distribution of outcomes into easy-to-read probabilities.

### Symbols
- $\Delta$ : relative return over horizon.
- $\varepsilon$ : flat zone threshold (e.g. 0.05 means ±5%).
- $|\Delta|$ : absolute value.

### Where it goes in the code
- `up_count = sum(Delta > eps)`
- `flat_count = sum(|Delta| <= eps)`
- `down_count = sum(Delta < -eps)`


---
## Final one-line summary

Monte Carlo here means:

1) simulate many daily log-return sequences $\ell_t$ (GBM or Bootstrap),
2) convert them to prices using $S_t = S_0\exp(\sum \ell)$,
3) analyze the resulting distribution using quantiles and probabilities.


In [ ]:
import numpy as np

# Core identity used in both GBM and Bootstrap simulations:
# Given simulated daily log returns `shocks` of shape (T, N):
# 1) cumulative sum over time gives Σ_{k<=t} ℓ_k for each scenario
# 2) exp converts log-change to multiplicative factor
# 3) multiply by S0 converts factor to price

def log_returns_to_prices(S0: float, shocks: np.ndarray) -> np.ndarray:
    """Convert log-return shocks (T, N) into price paths (T+1, N)."""
    log_paths = shocks.cumsum(axis=0)
    prices = S0 * np.exp(log_paths)
    prices = np.vstack([np.full((1, shocks.shape[1]), S0), prices])
    return prices
